# Employee Data Cleaning and Visualization

This notebook cleans the employee dataset, prints code outputs, exports `cleaned_data.csv`, and creates a bar plot, scatter plot, line plot, and histogram plot.

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

INPUT_FILE = Path('sample_data_cleaning_project - Sample_data_cleaning_project.csv')
OUTPUT_FILE = Path('cleaned_data.csv')
PLOT_DIR = Path('visualizations')
PLOT_DIR.mkdir(exist_ok=True)

data = pd.read_csv(INPUT_FILE)
print(f'Raw dataset shape: {data.shape}')
print('Missing values before cleaning:')
print(data.isna().sum())

In [2]:
duplicates_removed = int(data.duplicated().sum())
data = data.drop_duplicates().copy()
for column in data.select_dtypes(include='number').columns:
    data[column] = data[column].fillna(data[column].median())
for column in data.select_dtypes(exclude='number').columns:
    if data[column].isna().any():
        data[column] = data[column].fillna(data[column].mode().iloc[0])
data['Join_Date'] = pd.to_datetime(data['Join_Date'], errors='coerce')
data['Join_Date'] = data['Join_Date'].fillna(data['Join_Date'].min())
q1, q3 = data['Salary'].quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
before_outliers = len(data)
data = data[data['Salary'].between(lower, upper)].copy()
outliers_removed = before_outliers - len(data)
data = pd.get_dummies(data, columns=['Department'], dtype=int)
data.to_csv(OUTPUT_FILE, index=False)
print(f'Duplicate rows removed: {duplicates_removed}')
print(f'Salary outliers removed: {outliers_removed}')
print(f'Cleaned dataset shape: {data.shape}')
print('Missing values after cleaning:')
print(data.isna().sum())

In [3]:
print('Cleaned data preview:')
display(data.head())
print('Age and salary summary:')
display(data[['Age', 'Salary']].describe().round(2))

In [4]:
# Prepare readable categories for visualizations.
department_columns = [c for c in data.columns if c.startswith('Department_')]
chart_data = data.copy()
def department_name(row):
    for department in ('hr', 'it'):
        if row.get(f'Department_{department}', 0) == 1:
            return department.upper()
    return 'OTHER'
chart_data['Department'] = chart_data.apply(department_name, axis=1)
chart_data['Join_Year'] = pd.to_datetime(chart_data['Join_Date']).dt.year

# 1. Bar plot: employee count by department
department_counts = chart_data['Department'].value_counts().reindex(['HR', 'IT', 'OTHER'], fill_value=0)
ax = department_counts.plot(kind='bar', color=['#4C78A8', '#F58518', '#54A24B'], title='Employees by Department')
ax.set_xlabel('Department'); ax.set_ylabel('Employees'); plt.tight_layout()
plt.savefig(PLOT_DIR / 'bar_department_counts.png', dpi=150); plt.show(); plt.close()

# 2. Scatter plot: age and salary
plt.scatter(chart_data['Age'], chart_data['Salary'], c=chart_data['Department'].map({'HR':'#4C78A8','IT':'#F58518','OTHER':'#54A24B'}))
plt.title('Age and Salary Relationship'); plt.xlabel('Age'); plt.ylabel('Salary'); plt.grid(alpha=.25); plt.tight_layout()
plt.savefig(PLOT_DIR / 'scatter_age_salary.png', dpi=150); plt.show(); plt.close()

# 3. Line plot: average salary by joining year
year_salary = chart_data.groupby('Join_Year')['Salary'].mean()
plt.plot(year_salary.index, year_salary.values, marker='o', linewidth=2, color='#B279A2')
plt.title('Average Salary by Joining Year'); plt.xlabel('Join Year'); plt.ylabel('Average Salary'); plt.grid(alpha=.25); plt.tight_layout()
plt.savefig(PLOT_DIR / 'line_average_salary_by_year.png', dpi=150); plt.show(); plt.close()

# 4. Histogram: age distribution
plt.hist(chart_data['Age'], bins=8, color='#E45756', edgecolor='white')
plt.title('Employee Age Distribution'); plt.xlabel('Age'); plt.ylabel('Employees'); plt.tight_layout()
plt.savefig(PLOT_DIR / 'histogram_age_distribution.png', dpi=150); plt.show(); plt.close()

print(f'Saved four PNG visualizations to {PLOT_DIR.resolve()}')